# Load and Format Dataset

In [2]:
# imports
import numpy as np
import pandas as pd
import os

In [3]:
# load datasets
df_true = pd.read_csv('true.csv')
df_fake = pd.read_csv('fake.csv')

In [4]:
# peek true set
df_true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [5]:
# peek fake set
df_fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [6]:
# make labels
df_true['label'] = 0
df_fake['label'] = 1

In [7]:
df_fake.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23481 entries, 0 to 23480
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    23481 non-null  object
 1   text     23481 non-null  object
 2   subject  23481 non-null  object
 3   date     23481 non-null  object
 4   label    23481 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 917.4+ KB


In [8]:
df_true.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21417 entries, 0 to 21416
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    21417 non-null  object
 1   text     21417 non-null  object
 2   subject  21417 non-null  object
 3   date     21417 non-null  object
 4   label    21417 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 836.7+ KB


In [9]:
# check dimensions
print(r"Number of Rows and Column:",df_true.shape)
print(r"Number of Rows and Column:",df_fake.shape)

Number of Rows and Column: (21417, 5)
Number of Rows and Column: (23481, 5)


In [10]:
# label each dataset
df_true["label"]= 0
df_fake["label"]= 1

In [11]:
frame = [df_true, df_fake]

In [12]:
# concat both datasets
data = pd.concat(frame)
data.head()

,title,text,subject,date,label
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",0
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",0
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",0
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",0
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",0


In [13]:
# check concat success
data["label"].value_counts()

label
1    23481
0    21417
Name: count, dtype: int64

In [14]:
# create sentiment column: 0 -> real, 1 -> fake
def sentiment(label):
    if label == 0:
        return "real"
    else:
        return "fake"

In [15]:
# insert sentiment row based on label
data["sentiment"] = data["label"].apply(sentiment)

In [16]:
# verify sentiment column
data["sentiment"].value_counts()

sentiment
fake    23481
real    21417
Name: count, dtype: int64

# Data Preprocessing

In [17]:
# imports
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
ps = PorterStemmer()
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\11100\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [18]:
# read through each article entry and stem any words not considered stopwords
clean_df = []
stop_words = set(stopwords.words('english'))
for i in range(0, len(data)):
    review = data["text"].iloc[i]
    review = re.sub('[^a-zA-Z]', ' ', review)
    review = review.lower()
    review = review.split()
    review = [ps.stem(word) for word in review if not word in stop_words]
    review = ' '.join(review)
    clean_df.append(review)

In [19]:
# create TFidf Vectorizer
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,3))

In [20]:
X = tfidf.fit_transform(clean_df).toarray()
y = data["label"]

In [21]:
X_train, X_test, y_train, y_test, = train_test_split(X, y, train_size=0.7, random_state=67)

# Model Training

In [22]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
test_predict = model.predict(X_test)

# get fake probability
test_probs = model.predict_proba(X_test)[:, 1] * 100

In [23]:
# model eval
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

print(confusion_matrix(y_test, test_predict))
print(classification_report(y_test, test_predict))

score = accuracy_score(y_test, test_predict)
print("Accuracy: %0.3f" % score)

[[6369   66]
 [  96 6939]]
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      6435
           1       0.99      0.99      0.99      7035

    accuracy                           0.99     13470
   macro avg       0.99      0.99      0.99     13470
weighted avg       0.99      0.99      0.99     13470

Accuracy: 0.988


# Saving the Model and Vectorizer

In [24]:
import pickle
pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(tfidf, open("vectorizer.pkl", "wb"))

In [25]:
from sklearn.pipeline import Pipeline
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import StringTensorType

# create a pipeline to bundle vectorization and prediction
pipeline = Pipeline([
    ('tfidf', tfidf),
    ('model', model)
])

# convert to .onnx
initial_type = [('text_input', StringTensorType([None, 1]))]
onx = convert_sklearn(pipeline, initial_types=initial_type)
with open("model_pipeline.onnx", "wb") as f:
    f.write(onx.SerializeToString())